# Reproducing Fouke et al. (2025) from dandiset 001453

**Paper.** Fouke KE, He Z, Loring MD, Naumann EA (2025). Neural circuits underlying divergent
visuomotor strategies of zebrafish and *Danionella cerebrum*. *Current Biology*.
[doi:10.1016/j.cub.2025.04.027](https://doi.org/10.1016/j.cub.2025.04.027). This attempt used the
full text of the accepted author manuscript in PubMed Central (PMC12218792); the version of record
is not open access.

**Dandiset.** [DANDI:001453](https://dandiarchive.org/dandiset/001453/0.250518.1950), version
0.250518.1950. The paper cites no DANDI version (its data statement points to a GitLab repository
that holds code only). This is the only published version. The draft holds the same 37 files
(same blobs and sizes); it differs only in metadata added later (authors, keywords, and a
`relatedResource` link to the paper), none of which the analysis uses.

**What the dandiset holds.** Freely swimming tracking of 25 zebrafish (ZF) and 12 *D. cerebrum*
(DC) larvae during the optomotor protocol of Figure 1 and Figure S1. The two-photon imaging and the
head-fixed tail tracking of Figures 2 to 4 are not in it.

**Outside inputs.** The stimulus codes in the files have no names. The names come from the paper's
analysis notebook `Fig1-S1.ipynb` (GitLab `ean26/danionella_omr_neuralcircuits2025`, commit
33a5c29) and the protocol file of the stimulus software pandastim (GitHub
`Naumann-Lab/pandastim`, commit 7d9d161). The spatial scale (10 pixels per mm) is the factor in
`Fig1-S1.ipynb`. Claims marked "with inputs" use these and depend on the link claim.

No random numbers are drawn anywhere in this notebook.

| Claim | Data | Status |
|---|---|---|
| stimulus-code-link | with inputs | tested |
| fig1d-dc-faster-all-codes | dandiset | tested |
| fig1d-median-velocity | with inputs | tested |
| fig1d-zf-forward-faster-than-static | with inputs | tested |
| fig1d-dc-no-modulation | with inputs | tested |
| fig1d-dc-faster-oblique | with inputs | tested |
| fig1g-follow-direction | with inputs | tested |
| fig1h-zf-higher-angular-acceleration | with inputs | tested |
| figs1e-dc-respond-less-monocular | with inputs | tested |
| fig1c-continuous-vs-bouts | dandiset | tested |
| fig1cef-representative-values | dandiset | blocked: the representative fish and trials cannot be identified |
| fig2d, fig2g, fig2h, fig2k, fig2l (imaging) | dandiset | blocked: data missing |
| fig3e, fig3cd (head-fixed tail tracking) | dandiset | blocked: data missing |
| fig4f, fig4hi (imaging with tail tracking) | dandiset | blocked: data missing |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import omr

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = omr.load_all()
print(f"loaded {len(sessions)} files from dandiset {omr.DANDISET_ID} version {omr.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "fish": omr.fish_key(s), "species": s["species"], "subject_id": s["subject_id"], "age": s["age"],
    "session_start": s["session_start"][:19], "frames": s["n_frames"], "tracked": s["n_tracked"],
    "frame_hz": round(s["frame_hz"], 1), "x_max": round(s["x_range"][1]), "y_max": round(s["y_range"][1]),
    "has_dist": s["has_dist"], "same_timestamps": s["same_timestamps"]} for s in sessions])
print("Files per species:")
print(info.species.value_counts().to_string())
print("\nSubject ids used by both species:",
      sorted(set(info[info.species == "Danio rerio"].subject_id) & set(info[info.species != "Danio rerio"].subject_id), key=int))
print("Distinct subject ids:", info.subject_id.nunique(), "; distinct fish (species and id):", info.fish.nunique())
print("Age field values:", info.age.unique().tolist())
print("Paper, Figure 1D legend: N = ZF 26, DC 12 fish.")
print("\nEvery series in a file shares one timestamp array:", bool(info.same_timestamps.all()))
print("\nStated units:", sessions[0]["units"])
print("\n", info.sort_values(["species", "frame_hz"]).to_string(index=False))

The archive's summary says 25 subjects and lists only *Danio rerio*. The files hold 37 fish:
subject ids 0 to 11 are each used for one zebrafish and one *D. cerebrum*, so `subject_id` alone
does not identify an animal. Fish are named here by species and subject id. The 25 ZF are one fewer
than the 26 of the Figure 1D legend; the paper's notebook (`healthy_zf_fish`) lists 25 ZF and 12 DC,
and its ANOVA has 305 residual degrees of freedom, which fits 25 and 12 fish over nine stimuli
less ten empty cells. The age is given as a range (P5D/P9D) for every fish.

In [ ]:
fast = info.frame_hz > 100
print("Files by frame rate (Methods: tracking at more than 150 Hz):")
print(pd.crosstab(info.species, np.where(fast, "128-171 Hz", "32-35 Hz")).to_string())
dts = pd.DataFrame([{"fish": omr.fish_key(s), **{f"dt_{k}": v for k, v in s["dt_ms"].items()}} for s in sessions])
print("\nTimestamp intervals per file (ms), over all frames; within_10pct is the fraction of intervals within 10% of the median:")
print(dts.sort_values("dt_median").round(3).to_string(index=False))
print("Files whose median interval is below 6.67 ms (above 150 Hz):", int((dts.dt_median < 1000 / 150).sum()), "of", len(dts))

# What f0_vtheta is: compare its sum over a trial with the trial's change of f0_theta.
rows = []
for s in sessions:
    ratio_sum, ratio_int, vx_frame, vx_sec = [], [], [], []
    for number, c, idx in omr.trials(s):
        if len(idx) < 50:
            continue
        th, vth, t = s["f0_theta"][idx], s["f0_vtheta"][idx], s["t"][idx]
        if abs(th[-1] - th[0]) > 0.5:
            ratio_sum.append(vth.sum() / (th[-1] - th[0]))
            ratio_int.append(np.trapezoid(vth, t) / (th[-1] - th[0]))
        dx = np.diff(s["f0_x"][idx])
        vx = s["f0_vx"][idx][1:]
        ok = np.abs(dx) > 0.5
        if ok.sum() > 10:
            vx_frame.append(np.median(vx[ok] / dx[ok]))
            vx_sec.append(np.median(vx[ok] / (dx[ok] / np.diff(t)[ok])))
    rows.append({"fish": omr.fish_key(s), "frame_hz": round(s["frame_hz"]),
                 "sum_vtheta/dtheta": np.median(ratio_sum), "integral_vtheta_dt/dtheta": np.median(ratio_int),
                 "vx/(dx per frame)": np.median(vx_frame), "vx/(dx per s)": np.median(vx_sec)})
units = pd.DataFrame(rows)
print("\nMedian over trials, per file:")
print(units.round(3).to_string(index=False))
print("\nAcross files, median of sum(vtheta)/dtheta:", round(units["sum_vtheta/dtheta"].median(), 3),
      "; of integral(vtheta dt)/dtheta:", round(units["integral_vtheta_dt/dtheta"].median(), 4))

`f0_vtheta` is stated in radians/s. Summed over a trial's frames it gives the trial's change of
`f0_theta` (median ratio near 1 in most files), while its time integral gives about 1/frame rate
of it. It is the heading change per frame, in radians. The paper's code uses it that way (it sums
it to get the cumulative angle), so the analysis is unaffected, but the stated unit is wrong.
`f0_vx` and `f0_vy` are stated in pixels/s; the ratio to the per-frame displacement, printed above,
is the check: per frame it is 0.63 to 1.11, per second about the frame interval, so these are
pixels per frame. The paper's code converts them with the frame interval, which treats them as per
frame. They are not used below.

18 of 25 ZF files and 1 of 12 DC files are sampled at 32 to 35 Hz, against the Methods' "more than
150 Hz". The raw timestamp intervals show these are the rates of acquisition: in those files the
median interval is 28.8 to 30.8 ms and 96% to 99.6% of intervals are within 10% of it, and their
trials last a median 34.96 s (printed below), as the 35 s protocol has, so the timestamps are in
seconds. The other files have median intervals of 5.9 to 7.8 ms (128 to 171 Hz), quantised in
steps of about 1 ms, and only 11 of 37 files have a median interval below 6.67 ms (above 150 Hz).
The largest intervals in DC files (minutes to hours) are the gaps between trials. The ratio of `f0_vtheta`'s sum to the change of `f0_theta` is lower in most DC files
(0.09 to 0.89) than in most ZF files, so in DC files the two series do not agree over a trial; the
analysis uses `f0_vtheta`, as the paper's code does. The measures below are reported by frame-rate group where it could matter.

In [ ]:
# Spatial scale: trials start when the fish is in the central area, and end when it leaves the
# 45 mm active area. Positions are in pixels; no scale is given in the files.
rows = []
for s in sessions:
    starts = np.array([idx[0] for _, c, idx in omr.trials(s) if c != 20])
    inside = np.concatenate([idx for _, c, idx in omr.trials(s) if c != 20])
    x, y = s["f0_x"], s["f0_y"]
    rows.append({"fish": omr.fish_key(s),
                 "start_x_5_95": np.ptp(np.percentile(x[starts], [5, 95])),
                 "start_y_5_95": np.ptp(np.percentile(y[starts], [5, 95])),
                 "trial_x_1_99": np.ptp(np.percentile(x[inside], [1, 99])),
                 "trial_y_1_99": np.ptp(np.percentile(y[inside], [1, 99])),
                 "x_max": s["x_range"][1], "y_max": s["y_range"][1]})
extent = pd.DataFrame(rows).round(0)
print(extent.to_string(index=False))
print("\nMedian extent of trial start positions (px):", extent[["start_x_5_95", "start_y_5_95"]].median().to_dict())
print("Median extent of positions in trials (px):", extent[["trial_x_1_99", "trial_y_1_99"]].median().to_dict())
print("Files with any position beyond 1200 px:", extent[(extent.x_max > 1200) | (extent.y_max > 1200)].fish.tolist())

Trial start positions spread over about 500 pixels, and positions in trials over about 900 pixels,
with a maximum near 1100 in almost every file. At the code's 10 pixels per mm this is a start area
of about 50 mm and a field of about 110 mm, close to the 45 mm active area and the 100 mm dish of
the Methods. The scale is consistent with the data, not shown by it. One DC file has positions up
to about 1490 pixels, so its camera field, and possibly its scale, differs from the others.

In [ ]:
trial_rows = []
for s in sessions:
    for number, c, idx in omr.trials(s):
        t = s["t"][idx]
        trial_rows.append({"fish": omr.fish_key(s), "species": omr.fish_key(s)[:2], "code": c,
                           "trial": number, "duration_s": t[-1] - t[0]})
trial_table = pd.DataFrame(trial_rows)
print("Stimulus codes present:", sorted(trial_table.code.unique()))
counts = trial_table.pivot_table(index="fish", columns="code", values="trial", aggfunc="count", fill_value=0)
print("\nTrials per fish and stimulus code:")
print(counts.to_string())
print("\nFish without code 20 (static):", counts.index[counts[20] == 0].tolist())
blocks = pd.DataFrame([{"fish": omr.fish_key(s), "static_blocks": s["static_blocks"], "static_minutes": s["static_s"] / 60}
                       for s in sessions if s["static_blocks"]])
print("Code 20: trial numbers (stim_index) per fish:", blocks.static_blocks.value_counts().to_dict(),
      "; length of the block (min):", blocks.static_minutes.min().round(1), "to", blocks.static_minutes.max().round(1))
print("\nTrial duration (s) by species, codes 0-19 (code 20 is cut to its first 60 s on loading):")
print(trial_table[trial_table.code != 20].groupby("species").duration_s.describe().round(1).to_string())
zf_rate = trial_table[(trial_table.species == "ZF") & (trial_table.code != 20)].fish.map(lambda f: "128-171 Hz" if info.set_index("fish").frame_hz[f] > 100 else "32-35 Hz")
print("\nZF trial duration (s) by frame rate of the file (protocol: 35 s per trial):")
print(trial_table[(trial_table.species == "ZF") & (trial_table.code != 20)].groupby(zf_rate).duration_s.describe().round(2).to_string())
print("\nFraction of trial frames tracked (positions present), by species:")
print(info.assign(sp=info.fish.str[:2], frac=info.tracked / info.frames).groupby("sp").frac.describe().round(3).to_string())
print("\nFiles with a `dist` series:", info.groupby(info.fish.str[:2]).has_dist.sum().to_dict())

All files use codes 0 to 19, and 27 of 37 also code 20. The protocol file of the stimulus software
has 140 motion trials (20 stimuli, 7 repeats, each with a 3 s stationary phase and 35 s in all)
followed by a 4-hour "blank" block, which is code 20: the "static" condition of Figure 1D is the start of that block, one period per fish, after all motion trials (25 fish have one trial
number for code 20, two have two). `Fig1-S1.ipynb` handles code 20 like any trial number: its
`pass_criteria` keeps the first 35 s, and the measures use 3 to 35 s. The loader's 60 s cut keeps
that window, so static here is measured over the same frames as in the code, apart from the code's
filter on the DC `vel` column. 10 fish have no code 20. The
trials of DC are shorter than those of ZF (DC more often leave the active area, which ends a
trial). ZF files hold every frame, with untracked frames as NaN and frames between trials; DC files
hold only tracked trial frames, and carry an extra `dist` series. The two species were evidently
exported differently.

In [ ]:
# Recordings deposited twice: compare frame counts, the sequence of stimulus codes, and the first
# tracked positions of every pair of files.
sig = []
for s in sessions:
    seq = tuple(c for _, c, _ in omr.trials(s))
    xs = s["f0_x"][~np.isnan(s["f0_x"])][:500]
    sig.append((omr.fish_key(s), s["n_frames"], seq, xs))
pairs = []
for i in range(len(sig)):
    for j in range(i + 1, len(sig)):
        same_n = sig[i][1] == sig[j][1]
        same_seq = sig[i][2] == sig[j][2]
        n = min(len(sig[i][3]), len(sig[j][3]))
        same_x = n > 0 and np.allclose(sig[i][3][:n], sig[j][3][:n])
        if same_n or same_x or (same_seq and len(sig[i][2]) > 20):
            pairs.append((sig[i][0], sig[j][0], same_n, same_seq, same_x))
print("Pairs sharing frame count, stimulus sequence or first positions:", pairs if pairs else "none")
print("Distinct stimulus sequences:", len({s[2] for s in sig}), "of", len(sig))

No two files share a frame count or their first positions. The pairs listed share only the
sequence of stimulus codes: six ZF files ran the full fixed protocol of the stimulus software in
the same order. No recording appears to be deposited twice.

**Clock of the stimulus labels.** The stimulus code and trial number are series on the same
timestamps as the positions, so they share one clock by construction. Whether the labels are
aligned with what the fish saw is checked with a contrast: turning to code 3 minus turning to code 2
(right and left motion, by the outside mapping; the check itself does not need the names), in
0.5 s bins from the trial start. If the labels are aligned, the difference should be near zero in
the 3 s stationary phase and grow once the grating moves. With each trial given the code of the
trial before it, the difference should vanish.

In [ ]:
bins = np.arange(0, 20.5, 0.5)


def turning_course(shift):
    out = {}
    for s in sessions:
        tr = list(omr.trials(s))
        for k, (number, c, idx) in enumerate(tr):
            label = tr[k - shift][1] if k - shift >= 0 else None
            if label not in (2, 3):
                continue
            t = s["t"][idx] - s["t"][idx][0]
            deg = np.degrees(s["f0_vtheta"][idx])
            h = np.histogram(t, bins, weights=deg)[0] / 0.5
            out.setdefault((omr.fish_key(s)[:2], label), []).append(h)
    return out


rows = []
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, shift in zip(axes, (0, 1)):
    course = turning_course(shift)
    for sp, colour in (("ZF", "0.4"), ("DC", "k")):
        diff = np.mean(course[(sp, 3)], axis=0) - np.mean(course[(sp, 2)], axis=0)
        ax.plot(bins[:-1] + 0.25, diff, color=colour, label=sp)
        rows.append({"labels": "true" if shift == 0 else "previous trial", "species": sp,
                     "0-3 s": diff[bins[:-1] < 3].mean(), "3-20 s": diff[bins[:-1] >= 3].mean(),
                     "min": diff.min(), "max": diff.max()})
    ax.axvline(3, ls="--", color="0.6")
    ax.axhline(0, color="0.8")
    ax.set_title("true labels" if shift == 0 else "labels of the previous trial")
    ax.set_xlabel("time from trial start (s)")
axes[0].set_ylabel("turning, code 3 minus code 2 (deg/s)")
axes[0].legend()
plt.tight_layout()
plt.show()
print("Mean difference in turning (deg/s), code 3 minus code 2; axes autoscaled, all points shown:")
print(pd.DataFrame(rows).round(2).to_string(index=False))
course = turning_course(0)
zf_diff = np.mean(course[("ZF", 3)], axis=0) - np.mean(course[("ZF", 2)], axis=0)
print("\nZF, true labels, by 0.5 s bin from 0 to 6 s:", np.round(zf_diff[:12], 1).tolist())

For ZF the contrast with the true labels is 13.2 deg/s in the stationary phase and 34.8 deg/s from
3 to 20 s; in 0.5 s bins it falls to 2.6 and 5.9 deg/s between 2 and 3 s and rises from 3 s on,
to 40.2 deg/s at 4.5 to 5 s; with the labels of the previous trial it is 7.9
and 6.7 deg/s. The offset before 3 s is consistent with turning carried over from the previous
trial (in the fixed protocol, right motion follows medial-right motion, and the trials after a
code 3 trial start with a positive offset too). For ZF the stimulus labels are consistent with the
timing of the response. For DC the contrast is small (4.6 deg/s from 3 to 20 s with true labels,
0.1 with shifted labels) and noisier in the stationary phase (9.8 and 15.0 deg/s); the timing of
the DC labels could not be established by this check beyond that difference.

## Trial measures

Every trial is processed as in `Fig1-S1.ipynb` (see the helper's docstring). The counts of trials
dropped, by reason, follow.

In [ ]:
started = time.time()
measures = pd.concat([omr.trial_measures(s) for s in sessions], ignore_index=True)
measures["name"] = measures.code.map(omr.STIMULUS_NAMES)
hz = {omr.fish_key(s): s["frame_hz"] for s in sessions}
measures["rate"] = np.where(measures.fish.map(hz) > 100, "fast", "slow")
print(f"{len(measures)} trials measured in {time.time() - started:.0f} s")
print("\nTrials kept and dropped, by species and reason:")
print(measures.assign(reason=measures.reason.fillna("").replace("", "kept")).groupby(["species", "reason"]).size().to_string())
print("\nPassing trials per species:", measures[measures.passes].groupby("species").size().to_dict())
ok = measures[measures.passes]
print("Range of trial speed (mm/s) in passing trials:", ok.speed_mm_s.min().round(2), "to", ok.speed_mm_s.max().round(2))
print("Range of trial angular velocity (deg/s) in passing trials:", ok.angvel_deg_s.min().round(1), "to", ok.angvel_deg_s.max().round(1))
fast_trials = ok[ok.speed_mm_s > omr.MAX_SPEED_MM_S]
print("Passing trials whose measured speed exceeds 100 mm/s (the code's check uses 0 to 35 s and, for DC, "
      "the stored `dist`):", fast_trials.groupby("species").size().to_dict())
print("\nThose trials: the code's check speed from `dist`, the same check from smoothed positions, and the "
      "measured speed (3 to 35 s) and the length of the motion phase it is measured over (duration_s, s), mm/s:")
print(fast_trials[["fish", "code", "name", "trial", "n_frames", "duration_s", "check_speed_mm_s", "check_speed_pos_mm_s", "speed_mm_s"]]
      .round(1).to_string(index=False))
# Variant rejection: the speed check from smoothed positions in DC files too.
measures["passes_pos"] = measures.speed_mm_s.notna() & (measures.cum10_deg <= omr.MAX_CUM_ANGLE_10S) & \
    (measures.cum35_deg <= omr.MAX_CUM_ANGLE_35S) & (measures.check_speed_pos_mm_s <= omr.MAX_SPEED_MM_S)
print("\nTrials passing with the speed check from positions, by species:", measures[measures.passes_pos].groupby("species").size().to_dict(),
      "; passing one check and not the other:", measures[measures.passes != measures.passes_pos].groupby("species").size().to_dict())
dc_m = measures[(measures.species == "DC") & measures.check_speed_mm_s.notna()]
ratio_check = dc_m.check_speed_mm_s / dc_m.check_speed_pos_mm_s
print("DC trials, ratio of the `dist` check speed to the position check speed: median",
      round(ratio_check.median(), 3), "; 5th to 95th percentile", ratio_check.quantile([0.05, 0.95]).round(3).tolist())

Twelve passing trials have a measured speed above 100 mm/s, 11 of them DC. For DC the code's
check uses the stored `dist`, which does not agree with the positions (the ratio of the two check
speeds has a median of 1.11 and a 5th to 95th percentile of 0.31 to 3.79). Computing the check from
positions does not change which trials pass: the 11 DC trials have position-based check speeds of
31.4 to 96.5 mm/s, under the 100 mm/s limit. Their speed is high because it is measured over a short
motion phase (under 1 s in 9 of them). The exception is the DC-sub-2 static trial, 106.5 mm/s over
14.6 s. The code's check, taken over 0 to 35 s, does not catch these trials.

## stimulus-code-link

> The dandiset's stimulus codes can be linked to the stimuli the paper names, using the paper's
> analysis code and the protocol file of its stimulus software, and the values computed through
> that link agree with those the paper's code printed.

Reference values: the per-stimulus medians printed by `Fig1-S1.ipynb` (commit 33a5c29), cells
for Figure 1D (speed) and Figure 1G (angular velocity).

In [ ]:
CODE_PRINTED_SPEED = {  # mm/s, Fig1-S1.ipynb output, codes in WHOLE_FIELD order
    "DC": [11.9, 9.7, 10.0, 10.5, 10.5, 10.4, 8.8, 9.4, 9.0],
    "ZF": [5.1, 8.8, 8.4, 6.7, 4.9, 4.3, 5.2, 6.4, 8.3]}
CODE_PRINTED_ANGVEL = {  # deg/s
    "DC": [-1.8, -2.0, 1.8, 8.7, 3.4, 5.0, -2.3, 0.3, 1.5],
    "ZF": [2.3, 0.6, 6.6, 17.2, 7.7, 0.0, -21.5, -22.3, -1.4]}

used = [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 20]
present = set(trial_table.code.unique())
print("Codes the analysis uses that are missing from the dandiset:", sorted(set(used) - present) or "none")
print("Fish per species here:", info.fish.str[:2].value_counts().to_dict(),
      "; in Fig1-S1.ipynb's lists: ZF 25, DC 12; in the paper: ZF 26, DC 12")

speed_fish = omr.fish_medians(measures, "speed_mm_s")
angvel_fish = omr.fish_medians(measures, "angvel_deg_s")
rows = []
for sp in ("ZF", "DC"):
    for k, c in enumerate(omr.WHOLE_FIELD):
        sv = speed_fish[(speed_fish.species == sp) & (speed_fish.code == c)].speed_mm_s
        av = angvel_fish[(angvel_fish.species == sp) & (angvel_fish.code == c)].angvel_deg_s
        rows.append({"species": sp, "code": c, "name": omr.STIMULUS_NAMES[c], "n_fish": len(sv),
                     "speed_here": sv.median(), "speed_code": CODE_PRINTED_SPEED[sp][k],
                     "angvel_here": av.median(), "angvel_code": CODE_PRINTED_ANGVEL[sp][k]})
link = pd.DataFrame(rows)
link["speed_ratio"] = link.speed_here / link.speed_code
link["within_10pct"] = (link.speed_ratio - 1).abs() <= 0.10
print(link.round(2).to_string(index=False))
n_within = int(link.within_10pct.sum())
print(f"\nSpeed medians within 10% of the code's printed values: {n_within} of {len(link)}")
print("Median absolute difference in angular velocity (deg/s):",
      round((link.angvel_here - link.angvel_code).abs().median(), 2))
# Variant: Fig1-S1.ipynb sets fish medians of exactly 0 to NaN before its statistics and printed medians.
nz = speed_fish[speed_fish.speed_mm_s != 0]
print("\nFish x stimulus speed medians of exactly 0 (whole-field stimuli):",
      speed_fish[(speed_fish.speed_mm_s == 0) & speed_fish.code.isin(omr.WHOLE_FIELD)][["fish", "code"]].values.tolist())
link_nz = link[["species", "code", "name", "speed_code"]].copy()
link_nz["speed_here_no_zeros"] = [nz[(nz.species == r.species) & (nz.code == r.code)].speed_mm_s.median() for r in link_nz.itertuples()]
link_nz["within_10pct"] = (link_nz.speed_here_no_zeros / link_nz.speed_code - 1).abs() <= 0.10
print("With zero medians left out, speed medians within 10% of the printed values:", int(link_nz.within_10pct.sum()), "of 18;",
      "DC static", round(link_nz[(link_nz.species == "DC") & (link_nz.code == 20)].speed_here_no_zeros.iloc[0], 2), "mm/s")


In [ ]:
# Codes 9 and 11: the figure code calls 9 lateral right, the protocol conversion calls 11 lateral right.
rows = []
for c in (10, 13, 9, 11, 3, 2):
    for sp in ("ZF", "DC"):
        v = angvel_fish[(angvel_fish.species == sp) & (angvel_fish.code == c)].angvel_deg_s
        rows.append({"code": c, "protocol name": omr.STIMULUS_NAMES[c],
                     "figure-code name": omr.LATERAL_FROM_FIGURE_CODE.get(c, omr.STIMULUS_NAMES[c]),
                     "species": sp, "n_fish": len(v), "median_deg_s": v.median(),
                     "fish_turning_right": int((v > 0).sum())})
print(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
# Codes 9 and 11 against the medial codes 10 and 13: turning (deg/s) per fish, mean over trials in
# 4 s bins from the trial start (0-3 s is the stationary phase), passing trials only.
tbins = np.array([0, 3, 7, 11, 15, 19, 23, 27, 31, 35])
course_rows = []
passing_trials = set(zip(measures[measures.passes].path, measures[measures.passes].trial))
for s in sessions:
    for number, c, idx in omr.trials(s):
        if c not in (9, 10, 11, 13) or (s["path"], number) not in passing_trials:
            continue
        t = s["t"][idx] - s["t"][idx][0]
        deg = np.degrees(s["f0_vtheta"][idx])
        summed = np.histogram(t, tbins, weights=deg)[0]
        course_rows.append({"fish": omr.fish_key(s), "species": omr.fish_key(s)[:2], "code": c,
                            **{f"{a}-{b}s": v / (b - a) for a, b, v in zip(tbins[:-1], tbins[1:], summed)}})
courses = pd.DataFrame(course_rows)
cols = [c for c in courses.columns if c.endswith("s") and "-" in c]
per_fish_course = courses.groupby(["species", "fish", "code"])[cols].mean()
print("Species mean of fish means, by code (deg/s; positive is rightward):")
print(per_fish_course.groupby(["species", "code"]).mean().round(1).to_string())
sustained = per_fish_course[cols[1:]].mean(axis=1).unstack("code")
print("\nPer fish, mean turning from 3 to 35 s (deg/s), codes 9, 10, 11, 13:")
print(sustained.round(1).to_string())
for sp in ("ZF", "DC"):
    d = (sustained.loc[sp][9] - sustained.loc[sp][11]).dropna()
    print(f"{sp}: code 9 minus code 11, median {d.median():.2f} deg/s over {len(d)} fish, {int((d > 0).sum())} positive, "
          f"Wilcoxon p = {stats.wilcoxon(d).pvalue:.3g}; code 10 minus 13, median "
          f"{(sustained.loc[sp][10] - sustained.loc[sp][13]).dropna().median():.2f}")


**Result: reproduced.** Every code the analysis uses is in the dandiset. Through the link, 17 of
18 species and stimulus speed medians are within 10% of the values printed by the paper's code (the
exception is DC static, 10.52 against 11.9 mm/s), and the angular velocities differ from the
printed ones by a median of 0.02 deg/s. The fish counts (25 ZF, 12 DC) match the code's lists, not
the paper's 26 ZF. Fish cannot be matched one by one. No fish x stimulus speed median is exactly 0, so the
code's exclusion of zero medians changes nothing, and DC static stays at 10.52 mm/s. The static
window is the same as the code's (see the data check), which leaves the code's filter on the DC
`vel` column as the remaining difference. On codes 9 and 11, the ZF medians for both are near zero
(0.02 and 1.59 deg/s), while the medial codes 10 and 13 give 17.94 and -11.18 deg/s. Over 3 to 35 s,
DC turn more to the right for code 11 than for code 9 (median difference 5.82 deg/s, 10 of 12 fish,
Wilcoxon p = 0.0093); ZF go the same way but not significantly (1.66 deg/s, 16 of 25 fish,
p = 0.141). If the fish turn with lateral motion as they do with medial motion, this points to
code 11 being lateral right, as the protocol file has it. The data do not show that they do, so
the assignment is not settled.

## fig1d-dc-faster-all-codes

> *D. cerebrum* (DC) swim at a higher median velocity than zebrafish (ZF) across the visual
> stimuli.

Tested without the stimulus names, over all 21 codes, in pixels per second (no scale needed).

In [ ]:
def two_way(df, value, factor="code"):
    model = ols(f"{value} ~ C(species) + C({factor}) + C(species):C({factor})", data=df).fit()
    return sm.stats.anova_lm(model, typ=2)


all_codes = speed_fish.assign(speed_px_s=speed_fish.speed_mm_s * omr.PX_PER_MM)
table = two_way(all_codes, "speed_px_s")
print(table.to_string(float_format=lambda v: f"{v:.4g}"))
med = all_codes.groupby("species").speed_px_s.median()
print("\nMedian over fish and code values (px/s):", med.round(1).to_dict(),
      "; n values:", all_codes.groupby("species").size().to_dict())
per_fish = all_codes.groupby(["species", "fish"]).speed_px_s.median()
print("\nPer fish, median over codes (px/s):")
print(per_fish.round(1).to_string())
p_fish = stats.mannwhitneyu(per_fish.loc["DC"], per_fish.loc["ZF"]).pvalue
print(f"\nOne value per fish (median over codes): DC {per_fish.loc['DC'].median():.1f} against ZF {per_fish.loc['ZF'].median():.1f} px/s, "
      f"Mann-Whitney p = {p_fish:.2g} (the ANOVA above treats each fish x code value as independent)")


In [ ]:
# Variants: speed from positions resampled to 30 Hz; ZF split by frame rate.
s30 = omr.fish_medians(measures, "speed30_mm_s").assign(speed_px_s=lambda d: d.speed30_mm_s * omr.PX_PER_MM)
t30 = two_way(s30, "speed_px_s")
print("30 Hz variant, species effect: F =", round(t30.loc["C(species)", "F"], 2), "p =", f"{t30.loc['C(species)', 'PR(>F)']:.2g}",
      "; medians (px/s):", s30.groupby("species").speed_px_s.median().round(1).to_dict())
rate = all_codes.fish.map(lambda f: "fast" if hz[f] > 100 else "slow")
print("\nMedian over fish and code values (px/s) by species and frame rate:")
print(all_codes.assign(rate=rate).groupby(["species", "rate"]).speed_px_s.agg(["median", "size"]).round(1).to_string())
print("\nThe same with 30 Hz resampled positions:")
print(s30.assign(rate=s30.fish.map(lambda f: "fast" if hz[f] > 100 else "slow")).groupby(["species", "rate"]).speed_px_s.agg(["median", "size"]).round(1).to_string())

**Result: reproduced.** Over all 21 codes DC swim faster than ZF: median 94.0 against 55.2
pixels/s over fish and code values, species effect F = 78.4, p = 6.3e-18. With positions resampled
to 30 Hz the result is the same (95.2 against 53.8 pixels/s, p = 1.3e-08), and DC are faster within
each frame-rate group (fast files 86.4 against 59.9, slow files 106.6 against 52.9). The ANOVA treats each fish x code
value as independent; with one value per fish (the median over codes), DC 87.4 against ZF 52.9
pixels/s, Mann-Whitney p = 0.02.

## fig1d-median-velocity

> Across the whole-field stimuli DC have a higher median velocity than ZF (9.8 against 6.4 mm/s),
> with a significant main effect of species in a two-way ANOVA.

In [ ]:
wf = speed_fish[speed_fish.code.isin(omr.WHOLE_FIELD)].copy()
wf["name"] = wf.code.map(omr.STIMULUS_NAMES)
wide = wf.pivot_table(index=["species", "fish"], columns="name", values="speed_mm_s")
print("Per fish median speed (mm/s), whole-field stimuli:")
print(wide.round(1).to_string())


def readings(fish_values, trials_values, column):
    out = {}
    for sp in ("DC", "ZF"):
        f = fish_values[fish_values.species == sp]
        out[sp] = {
            "a: median of stimulus medians": f.groupby("code")[column].median().median(),
            "b: median of fish x stimulus values": f[column].median(),
            "c: median over fish of fish medians": f.groupby("fish")[column].median().median(),
            "d: median of passing trials": trials_values[trials_values.species == sp][column].median()}
    return pd.DataFrame(out)


wf_trials = measures[measures.passes & measures.code.isin(omr.WHOLE_FIELD)]
read = readings(wf, wf_trials, "speed_mm_s")
read["DC within 9.8 +/- 0.98"] = (read.DC - 9.8).abs() <= 0.98
read["ZF within 6.4 +/- 0.64"] = (read.ZF - 6.4).abs() <= 0.64
print("\nOverall medians (mm/s) under each reading:")
print(read.round(2).to_string())
anova_wf = two_way(wf, "speed_mm_s")
print("\nTwo-way ANOVA (type II) on fish x stimulus medians:")
print(anova_wf.to_string(float_format=lambda v: f"{v:.4g}"))

In [ ]:
# Variants: 30 Hz speeds; no trial rejection; by ZF frame-rate group.
wf30 = s30[s30.code.isin(omr.WHOLE_FIELD)]
r30 = readings(wf30, measures[measures.passes & measures.code.isin(omr.WHOLE_FIELD)], "speed30_mm_s")
a30 = two_way(wf30, "speed30_mm_s")
print("30 Hz positions: overall medians (mm/s)")
print(r30.round(2).to_string())
print("species effect p =", f"{a30.loc['C(species)', 'PR(>F)']:.2g}")

no_reject = measures.copy()
no_reject["passes"] = no_reject.speed_mm_s.notna()
nr = omr.fish_medians(no_reject, "speed_mm_s")
nr = nr[nr.code.isin(omr.WHOLE_FIELD)]
print("\nWithout the code's trial rejection (trials with a measure:", int(no_reject.passes.sum()), "):")
print(readings(nr, no_reject[no_reject.passes & no_reject.code.isin(omr.WHOLE_FIELD)], "speed_mm_s").round(2).to_string())
print("species effect p =", f"{two_way(nr, 'speed_mm_s').loc['C(species)', 'PR(>F)']:.2g}")

wf_rate = wf.assign(rate=wf.fish.map(lambda f: "fast" if hz[f] > 100 else "slow"))
print("\nReading (b) by species and frame rate (mm/s):")
print(wf_rate.groupby(["species", "rate"]).speed_mm_s.agg(["median", "size"]).round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for k, c in enumerate(omr.WHOLE_FIELD):
    for dx, sp, colour in ((-0.15, "ZF", "0.5"), (0.15, "DC", "k")):
        v = wf[(wf.species == sp) & (wf.code == c)].speed_mm_s
        ax.scatter(np.full(len(v), k + dx), v, s=10, color=colour, label=sp if k == 0 else None)
        ax.hlines(v.median(), k + dx - 0.1, k + dx + 0.1, color="r")
ax.set_xticks(range(len(omr.WHOLE_FIELD)), [omr.STIMULUS_NAMES[c] for c in omr.WHOLE_FIELD], rotation=30)
ax.set_ylabel("fish median speed (mm/s)")
ax.legend()
plt.tight_layout()
plt.show()
print("Plotted range (mm/s):", wf.speed_mm_s.min().round(2), "to", wf.speed_mm_s.max().round(2), "; axes autoscaled, all points shown")

**Result: reproduced.** Under every reading both overall medians are within tolerance of 9.8
and 6.4 mm/s: reading (a) gives DC 10.02 and ZF 6.39, (b) 9.87 and 6.41, (c) 9.36 and 6.33, (d)
9.88 and 6.28. The species effect of the two-way ANOVA is F = 20.8, p = 7.5e-06. With positions
resampled to 30 Hz, DC give 9.43 to 10.15 and ZF 6.07 to 6.30, still within tolerance (p = 0.0011);
without the code's trial rejection (4487 trials in place of 4458) the
readings are unchanged except (d), DC 9.85 and ZF 6.30, and p = 6.7e-06. ZF from 32 to 35 Hz files are slower
(5.88 mm/s, reading b) than ZF from 167 to 171 Hz files (6.98).

## fig1d-zf-forward-faster-than-static

> ZF modulate their velocity with the direction of motion; forward motion increases their median
> velocity (static 5.0 mm/s, forward 8.8 mm/s).

In [ ]:
for c in (20, 0):
    v = wf[(wf.species == "ZF") & (wf.code == c)].speed_mm_s
    print(f"ZF {omr.STIMULUS_NAMES[c]:8s} median {v.median():.2f} mm/s over {len(v)} fish "
          f"(interquartile {v.quantile(0.25):.2f} to {v.quantile(0.75):.2f})")
paired = wide.loc["ZF"][["static", "forward"]].dropna()
print(f"\nZF fish with both: {len(paired)}; forward faster in {(paired.forward > paired.static).sum()}; "
      f"median of forward minus static {np.median(paired.forward - paired.static):.2f} mm/s; "
      f"Wilcoxon p = {stats.wilcoxon(paired.forward, paired.static).pvalue:.3g}")
print("\nBy ZF frame-rate group (median mm/s, n fish):")
print(wf_rate[(wf_rate.species == "ZF") & wf_rate.code.isin([20, 0])].groupby(["rate", "code"]).speed_mm_s.agg(["median", "size"]).round(2).to_string())
v30 = wf30[(wf30.species == "ZF")].groupby("code").speed30_mm_s.median()
print("\n30 Hz variant, ZF static and forward (mm/s):", round(v30[20], 2), round(v30[0], 2))

**Result: reproduced.** ZF median speed is 4.84 mm/s for static (18 fish) and 8.78 mm/s for
forward (25 fish), against 5.0 and 8.8. Of 18 fish with both, 14 are faster to forward motion
(median difference 3.67 mm/s, Wilcoxon p = 0.0077). The static value depends on frame rate: 8.31
mm/s in the 5 fish from fast files, 4.50 in the 13 from slow files; forward does not (8.94 and
8.69).

## fig1d-dc-no-modulation

> DC lack the direction-dependent velocity modulation seen in ZF.

The paper's test: Tukey's HSD over all 18 species and stimulus groups of fish medians.

In [ ]:
groups = wf.species + "_" + wf.name
tukey = pairwise_tukeyhsd(wf.speed_mm_s, groups, alpha=0.05)
res = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
dc_pairs = res[res.group1.str.startswith("DC") & res.group2.str.startswith("DC")]
zf_pairs = res[res.group1.str.startswith("ZF") & res.group2.str.startswith("ZF")]
print(f"DC pairs: {len(dc_pairs)}, significant: {int(dc_pairs.reject.sum())}; smallest adjusted p {dc_pairs['p-adj'].min():.3f}")
print(f"ZF pairs: {len(zf_pairs)}, significant: {int(zf_pairs.reject.sum())}; smallest adjusted p {zf_pairs['p-adj'].min():.3f}")
print(res[res.reject].to_string(index=False) if res.reject.any() else "no significant pair")
dc_med = wf[wf.species == "DC"].groupby("name").speed_mm_s.median()
zf_med = wf[wf.species == "ZF"].groupby("name").speed_mm_s.median()
print(f"\nRange of DC per-stimulus medians: {dc_med.min():.2f} to {dc_med.max():.2f} mm/s (ratio {dc_med.max() / dc_med.min():.2f})")
print(f"Range of ZF per-stimulus medians: {zf_med.min():.2f} to {zf_med.max():.2f} mm/s (ratio {zf_med.max() / zf_med.min():.2f})")
for sp in ("DC", "ZF"):
    sub = wf[wf.species == sp]
    print(f"{sp}: one-way ANOVA across the nine stimuli, p = {stats.f_oneway(*[g.speed_mm_s for _, g in sub.groupby('code')]).pvalue:.3g}; "
          f"Friedman over fish with all nine: ", end="")
    w = sub.pivot_table(index="fish", columns="code", values="speed_mm_s").dropna()
    print(f"{len(w)} fish, p = {stats.friedmanchisquare(*[w[c] for c in w.columns]).pvalue:.3g}" if len(w) > 2 else "too few")
print("\nDC static, per fish (mm/s):", wf[(wf.species == 'DC') & (wf.code == 20)].set_index('fish').speed_mm_s.round(1).to_dict())

In [ ]:
CODE_PRINTED_TUKEY_DC = {  # Fig1-S1.ipynb, Tukey over the 18 groups: DC pairs with adjusted p < 0.05
    ("backward_left", "static"): 0.0236, ("backward_right", "static"): 0.0373, ("forward_left", "static"): 0.0397}
here = {tuple(sorted((r.group1[3:], r.group2[3:]))): r["p-adj"] for _, r in dc_pairs[dc_pairs.reject].iterrows()}
print("DC pairs significant in Fig1-S1.ipynb's printed output:", CODE_PRINTED_TUKEY_DC)
print("DC pairs significant here:", {k: round(v, 4) for k, v in here.items()})


def tukey_counts(sub):
    t = pairwise_tukeyhsd(sub.speed_mm_s, sub.species + "_" + sub.name)
    r = pd.DataFrame(t.summary().data[1:], columns=t.summary().data[0])
    dcp = r[r.group1.str.startswith("DC") & r.group2.str.startswith("DC")]
    zfp = r[r.group1.str.startswith("ZF") & r.group2.str.startswith("ZF")]
    return (f"DC pairs significant {int(dcp.reject.sum())} of {len(dcp)} (smallest p {dcp['p-adj'].min():.3f}); "
            f"ZF pairs significant {int(zfp.reject.sum())} of {len(zfp)} (smallest p {zfp['p-adj'].min():.3f})")


# Variants: the eight motion directions only; fish-stimulus values above 100 mm/s left out; fish medians
# of exactly 0 left out, as Fig1-S1.ipynb does; the speed check computed from positions in DC files too.
pos = omr.fish_medians(measures.assign(passes=measures.passes_pos), "speed_mm_s")
pos = pos[pos.code.isin(omr.WHOLE_FIELD)].assign(name=lambda d: d.code.map(omr.STIMULUS_NAMES))
for label, sub in (("directions only", wf[wf.code != 20]), ("values above 100 mm/s left out", wf[wf.speed_mm_s <= 100]),
                   ("zero medians left out", wf[wf.speed_mm_s != 0]), ("speed check from positions", pos)):
    print(f"\n{label}: {tukey_counts(sub)}")
print("\nDC static, per fish, speed check from positions (mm/s):", pos[(pos.species == 'DC') & (pos.code == 20)].set_index('fish').speed_mm_s.round(1).to_dict())


**Result: not reproduced.** By the paper's test, Tukey's HSD over the 18 groups, 3 of 36 DC
pairs differ (static against backward left, backward right and forward left, adjusted p 0.030 to
0.049), so the test finds a difference within DC. All three involve static, whose DC values come
from 9 fish and include one at 106.5 mm/s and two at 0.0. Among the eight motion directions no DC
pair differs (smallest adjusted p 0.999); the DC per-stimulus medians span 8.81 to 10.52 mm/s
(ratio 1.19, against 2.03 in ZF), and a one-way ANOVA across the nine DC stimuli gives p = 0.479
(ZF p = 0.00082). With values above 100 mm/s left out, no DC pair differs. The same Tukey test
finds no ZF pair over the 18 groups (smallest adjusted p 0.835), and 3 of 28 among the directions
alone. The output of `Fig1-S1.ipynb` itself shows the same three DC pairs as significant (adjusted p
0.0236, 0.0373 and 0.0397), so the paper's own test disagrees with the claim, and the result here
matches the paper's computation. Computing the speed check from positions, or leaving out zero
medians as the code does, gives the same 3 pairs. The DC-sub-2 static value of 106.5 mm/s comes from
one trial (see the trial measures), and without values above 100 mm/s no DC pair differs.

## fig1d-dc-faster-oblique

> DC swim at a significantly higher velocity than ZF for oblique stimuli.

In [ ]:
rows = []
for c in omr.WHOLE_FIELD:
    sub = wf[wf.code == c]
    t = pairwise_tukeyhsd(sub.speed_mm_s, sub.species)
    rows.append({"code": c, "name": omr.STIMULUS_NAMES[c],
                 "DC_median": sub[sub.species == "DC"].speed_mm_s.median(),
                 "ZF_median": sub[sub.species == "ZF"].speed_mm_s.median(),
                 "mean_DC_minus_ZF": sub[sub.species == "DC"].speed_mm_s.mean() - sub[sub.species == "ZF"].speed_mm_s.mean(),
                 "p": t.pvalues[0], "n_DC": int((sub.species == "DC").sum()), "n_ZF": int((sub.species == "ZF").sum())})
per_stim = pd.DataFrame(rows)
per_stim["significant_at_0.05/9"] = per_stim.p < 0.05 / 9
print(per_stim.round(4).to_string(index=False))
reading_a = ["forward_right", "forward_left", "backward_right", "backward_left"]
reading_b = ["right", "left", "backward_right", "backward_left"]
for label, names in (("a, intercardinal", reading_a), ("b, 90 degrees or more", reading_b)):
    sel = per_stim[per_stim.name.isin(names)]
    n = int((sel["significant_at_0.05/9"] & (sel.mean_DC_minus_ZF > 0)).sum())
    print(f"Reading {label}: {n} of 4 stimuli with DC significantly faster")

**Result: reproduced.** Under reading (b) DC are significantly faster than ZF for right
(p = 0.0002) and backward right (p = 0.0047), 2 of 4; under reading (a) for 1 of 4 (backward
right). Backward motion, which is not oblique, also differs (p = 0.0002).

## fig1g-follow-direction

> Both species follow the direction of optic flow, turning right (positive angular velocity) to
> rightward motion and left to leftward motion.

In [ ]:
awf = angvel_fish[angvel_fish.code.isin(omr.WHOLE_FIELD)].copy()
awf["name"] = awf.code.map(omr.STIMULUS_NAMES)
print("Species medians of fish median angular velocity (deg/s), and fish turning right:")
summary = awf.groupby(["name", "species"]).angvel_deg_s.agg(["median", "size", lambda v: int((v > 0).sum())])
summary.columns = ["median", "n_fish", "n_positive"]
print(summary.round(2).unstack("species").to_string())

aw = awf.pivot_table(index=["species", "fish"], columns="code", values="angvel_deg_s")
score = (aw[omr.RIGHTWARD].mean(axis=1) - aw[omr.LEFTWARD].mean(axis=1)).rename("right_minus_left")
print("\nPer fish, mean over rightward minus mean over leftward stimuli (deg/s):")
print(score.round(2).to_string())
for sp in ("ZF", "DC"):
    v = score.loc[sp].dropna()
    p = stats.wilcoxon(v, alternative="greater").pvalue
    print(f"{sp}: {len(v)} fish, median {v.median():.2f} deg/s, {int((v > 0).sum())} positive, one-sided Wilcoxon p = {p:.3g}")

In [ ]:
# Variant: by frame-rate group, and the binocular pair alone (right minus left).
rate_aw = score.reset_index().assign(rate=lambda d: d.fish.map(lambda f: "fast" if hz[f] > 100 else "slow"))
print(rate_aw.groupby(["species", "rate"]).right_minus_left.agg(["median", "size"]).round(2).to_string())
pair = (aw[3] - aw[2]).rename("right_minus_left_binocular")
for sp in ("ZF", "DC"):
    v = pair.loc[sp].dropna()
    print(f"{sp}, codes 3 minus 2 only: median {v.median():.2f} deg/s over {len(v)} fish, p = {stats.wilcoxon(v, alternative='greater').pvalue:.3g}")

**Result: partially reproduced.** ZF follow the direction of motion: the rightward minus leftward
difference is positive in 23 of 25 fish (median 29.51 deg/s, one-sided Wilcoxon p = 9.8e-07). DC do
not by this test: 7 of 12 fish are positive, median 0.88 deg/s, p = 0.151, and the DC median for
left motion is +0.25 deg/s. Codes 3 minus 2 alone give the same (ZF 45.08 deg/s, p = 2.6e-06; DC
3.10 deg/s, p = 0.133). DC turn right to right motion (median 8.67 deg/s), and four DC fish have
differences above 12 deg/s.

## fig1h-zf-higher-angular-acceleration

> Across directional stimuli ZF have higher angular acceleration than DC, and the distributions
> of the two species differ for every stimulus.

In [ ]:
started = time.time()
passing = set(zip(measures[measures.passes].path, measures[measures.passes].trial))
accel = pd.concat([omr.acceleration_frames(s, passing) for s in sessions], ignore_index=True)
accel = accel[accel.code.isin(omr.WHOLE_FIELD)]
print(f"angular acceleration frames computed in {time.time() - started:.0f} s; frames kept:")
print(accel.groupby(["kind", "species"]).size().to_string())


def accel_tests(kind):
    rows = []
    for c in omr.WHOLE_FIELD:
        sub = accel[(accel.kind == kind) & (accel.code == c)]
        z, d = sub[sub.species == "ZF"].accel, sub[sub.species == "DC"].accel
        fz = sub[sub.species == "ZF"].groupby("fish").accel.median()
        fd = sub[sub.species == "DC"].groupby("fish").accel.median()
        rows.append({"name": omr.STIMULUS_NAMES[c], "ZF_median": z.median(), "DC_median": d.median(),
                     "n_ZF": len(z), "n_DC": len(d), "kruskal_p": stats.kruskal(z, d).pvalue,
                     "ZF_fish_median": fz.median(), "DC_fish_median": fd.median(),
                     "fish_mannwhitney_p": stats.mannwhitneyu(fz, fd).pvalue})
    out = pd.DataFrame(rows)
    out["zf_higher_and_sig"] = (out.ZF_median > out.DC_median) & (out.kruskal_p < 0.05 / 9)
    return out


main = accel_tests("methods")
print("\nMethods definition (deg/s^2, heading at 30 Hz), frames 10 to 1000:")
print(main.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"\nStimuli with ZF higher and Kruskal-Wallis p < 0.05/9: {int(main.zf_higher_and_sig.sum())} of 9")
print(f"Per-fish variant, ZF fish median higher and Mann-Whitney p < 0.05/9: "
      f"{int(((main.ZF_fish_median > main.DC_fish_median) & (main.fish_mannwhitney_p < 0.05 / 9)).sum())} of 9")

In [ ]:
coded = accel_tests("code")
print("Variant: Fig1-S1.ipynb's quantity (per-frame heading change / file mean frame interval):")
print(coded.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"\nStimuli with ZF higher and p < 0.05/9: {int(coded.zf_higher_and_sig.sum())} of 9")
ratio = pd.DataFrame([{"fish": omr.fish_key(s), "frame_interval_median_ms": 1000 / s["frame_hz"], "code_dt_ms": 1000 * s["code_dt"]} for s in sessions])
ratio["code_dt_over_frame_interval"] = ratio.code_dt_ms / ratio.frame_interval_median_ms
fast_fish = {f for f, v in hz.items() if v > 100}
sub = accel[(accel.kind == "methods") & accel.fish.isin(fast_fish)]
fm = sub.groupby(["species", "fish"]).accel.median()
print("\nMethods definition, files at 128-171 Hz only: per-fish median over all whole-field frames (deg/s^2):",
      f"ZF {fm.loc['ZF'].median():.1f} (n {len(fm.loc['ZF'])}), DC {fm.loc['DC'].median():.1f} (n {len(fm.loc['DC'])}),",
      f"Mann-Whitney p = {stats.mannwhitneyu(fm.loc['ZF'], fm.loc['DC']).pvalue:.3g}")
print("\nThe code's mean interval against the median frame interval, by species:")
print(ratio.groupby(ratio.fish.str[:2]).code_dt_over_frame_interval.describe().round(2).to_string())
# Noise floor: the stationary phase (0 to 3 s, grating still) against the motion phase, and the
# measure from the unwrapped f0_theta in place of the cumulative sum of f0_vtheta.
floor = accel[accel.kind.isin(["methods", "theta"])].groupby(["kind", "species", "stationary"]).accel.agg(["median", "size"])
print("\nMedian absolute angular acceleration (deg/s^2) by phase (stationary = first 3 s), frames 10 to 1000:")
print(floor.round(1).to_string())
fish_floor = accel[accel.kind == "methods"].groupby(["species", "fish", "stationary"]).accel.median().unstack()
fish_floor.columns = ["motion", "stationary"]
print("\nPer fish (Methods definition): median in the stationary phase against the motion phase:")
print(fish_floor.round(1).to_string())
for sp in ("ZF", "DC"):
    f = fish_floor.loc[sp].dropna()
    print(f"{sp}: stationary over motion, median ratio {(f.stationary / f.motion).median():.2f} over {len(f)} fish; "
          f"Wilcoxon p = {stats.wilcoxon(f.stationary, f.motion).pvalue:.3g}")
theta = accel_tests("theta")
print("\nVariant: heading from the unwrapped f0_theta:")
print(theta[["name", "ZF_median", "DC_median", "kruskal_p", "ZF_fish_median", "DC_fish_median", "fish_mannwhitney_p"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"Stimuli with ZF higher and p < 0.05/9: {int(theta.zf_higher_and_sig.sum())} of 9")


**Result: not reproduced.** With angular acceleration as the Methods define it, DC are higher
than ZF for all nine stimuli (medians 143 to 191 against 55.4 to 66.4 deg/s², Kruskal-Wallis p
below 0.05/9 for each), the reverse of the claim, so 0 of 9 stimuli meet the criterion. Per-fish
medians give the same, and so do the files at 128 to 171 Hz alone (ZF 84.9, DC 178.6 deg/s²,
p = 6.3e-05). The quantity Fig1-S1.ipynb computes gives ZF higher with p below 0.05/9 for 7 of 9
stimuli. That quantity divides each frame's heading change by the file's mean frame interval, which
in DC files is a median 8.73 times the frame interval (ZF 1.37), because DC files hold only trial
frames and the gaps between trials enter the mean.

The DC values may be a tracking-noise floor, not turning. In the stationary phase (first 3 s, grating
still) the DC median is 205 deg/s², higher than the 171 during motion (per fish, a median ratio of
1.24, Wilcoxon p = 0.016); ZF give 66.9 and 60.8 (ratio 1.07). With the heading from the unwrapped
`f0_theta` in place of the summed `f0_vtheta`, DC are higher still (218 to 278 against ZF 61.8 to
77.3 deg/s², 0 of 9 stimuli meet the criterion). Both heading series give DC the higher values, and
neither shows a DC response to motion, so the reversal should not be read as evidence against the
paper.

## figs1e-dc-respond-less-monocular

> DC generally respond less than ZF to monocular stimuli, suggesting that they need more visual
> motion input to drive the optomotor response.

In [ ]:
def monocular_table(lateral_right, lateral_left):
    cats = {"static": [(20, 1)], "medial": [(10, 1), (13, -1)],
            "lateral": [(lateral_right, 1), (lateral_left, -1)], "binocular": [(3, 1), (2, -1)]}
    rows = []
    for cat, members in cats.items():
        for c, sign in members:
            sub = angvel_fish[angvel_fish.code == c]
            for _, r in sub.iterrows():
                rows.append({"category": cat, "species": r.species, "fish": r.fish, "code": c,
                             "toward_deg_s": sign * r.angvel_deg_s})
    return pd.DataFrame(rows)


def monocular_tests(df):
    model = ols("toward_deg_s ~ C(species) + C(category) + C(species):C(category)", data=df).fit()
    table = sm.stats.anova_lm(model, typ=2)
    rows = []
    for cat in ("static", "medial", "lateral", "binocular"):
        sub = df[df.category == cat]
        t = pairwise_tukeyhsd(sub.toward_deg_s, sub.species)
        rows.append({"category": cat, "ZF_median": sub[sub.species == "ZF"].toward_deg_s.median(),
                     "DC_median": sub[sub.species == "DC"].toward_deg_s.median(),
                     "n_ZF": int((sub.species == "ZF").sum()), "n_DC": int((sub.species == "DC").sum()),
                     "p": t.pvalues[0], "significant_at_0.05/4": t.pvalues[0] < 0.05 / 4})
    return table, pd.DataFrame(rows)


mono = monocular_table(9, 11)
table, per_cat = monocular_tests(mono)
print("Figure-code assignment (9 lateral right, 11 lateral left). Two-way ANOVA (type II):")
print(table.to_string(float_format=lambda v: f"{v:.4g}"))
print()
print(per_cat.round(4).to_string(index=False))
print("\nPer fish, angular velocity toward the stimulus (deg/s), mean of the two sides:")
print(mono.groupby(["species", "fish", "category"]).toward_deg_s.mean().unstack().round(1).to_string())

In [ ]:
swapped = monocular_table(11, 9)
t2, p2 = monocular_tests(swapped)
print("Variant, protocol-file assignment (11 lateral right, 9 lateral left):")
print("species effect p =", f"{t2.loc['C(species)', 'PR(>F)']:.3g}")
print(p2.round(4).to_string(index=False))

**Result: reproduced.** The species effect is significant (F = 23.3, p = 2.5e-06), and
DC turn less toward the stimulus than ZF for medial stimuli (median 3.26 against 13.48 deg/s,
p = 0.0003) and binocular ones (2.87 against 21.87 deg/s, p = 0.0002). For lateral stimuli DC are
below ZF only nominally (-1.07 against -0.13 deg/s, p = 0.96): neither species turns toward lateral
motion, and the criterion's lateral part holds by 0.95 deg/s. With the protocol file's assignment
of codes 9 and 11 the lateral medians change sign (ZF 0.13, DC 1.07 deg/s), DC are no longer below
ZF there, and that variant would be partially reproduced. The DC turning time courses (under the
link claim) lean toward the protocol file's assignment if the fish turn with lateral motion, so the
lateral part of this result is fragile.

## fig1c-continuous-vs-bouts

> ZF swim in discrete bouts separated by glides, while DC swim continuously at a relatively
> constant velocity.

Measure fixed in the claims file: the ratio of median to mean instantaneous speed (positions at
30 Hz, 3 to 35 s), per fish the median over all trials of all codes.

In [ ]:
steady = measures[measures.passes].groupby(["species", "fish"]).steadiness.median()
print("Per fish median steadiness (median / mean instantaneous speed):")
print(steady.round(3).to_string())
z, d = steady.loc["ZF"].dropna(), steady.loc["DC"].dropna()
p = stats.mannwhitneyu(z, d, alternative="two-sided").pvalue
print(f"\nZF median {z.median():.3f} over {len(z)} fish; DC median {d.median():.3f} over {len(d)} fish; Mann-Whitney p = {p:.3g}")
by_rate = steady.reset_index().assign(rate=lambda df: df.fish.map(lambda f: "fast" if hz[f] > 100 else "slow"))
print("\nBy frame rate of the file:")
print(by_rate.groupby(["species", "rate"]).steadiness.agg(["median", "size"]).round(3).to_string())
zs = by_rate[(by_rate.species == "ZF") & (by_rate.rate == "fast")].steadiness
ds = by_rate[(by_rate.species == "DC") & (by_rate.rate == "fast")].steadiness
print(f"Fast files only: ZF {zs.median():.3f} (n {len(zs)}) against DC {ds.median():.3f} (n {len(ds)}), p = {stats.mannwhitneyu(zs, ds).pvalue:.3g}")
fwd = measures[measures.passes & (measures.code == 0)].groupby(["species", "fish"]).steadiness.median()
print(f"Forward trials only (code 0): ZF {fwd.loc['ZF'].median():.3f}, DC {fwd.loc['DC'].median():.3f}, p = {stats.mannwhitneyu(fwd.loc['ZF'].dropna(), fwd.loc['DC'].dropna()).pvalue:.3g}")

**Result: reproduced.** The median-to-mean ratio of instantaneous speed is 0.306 in ZF (25 fish,
0.244 to 0.580) and 0.908 in DC (12 fish, 11 of them above 0.74 and one at 0.140), Mann-Whitney
p = 5.4e-05. The difference holds in the files at 128 to 171 Hz alone (0.298 against 0.898,
p = 0.0028) and in forward trials alone (0.376 against 0.956, p = 1.2e-06).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| stimulus-code-link | with inputs | reproduced |
| fig1d-dc-faster-all-codes | dandiset | reproduced |
| fig1d-median-velocity | with inputs | reproduced |
| fig1d-zf-forward-faster-than-static | with inputs | reproduced |
| fig1d-dc-no-modulation | with inputs | not reproduced |
| fig1d-dc-faster-oblique | with inputs | reproduced |
| fig1g-follow-direction | with inputs | partially reproduced |
| fig1h-zf-higher-angular-acceleration | with inputs | not reproduced |
| figs1e-dc-respond-less-monocular | with inputs | reproduced |
| fig1c-continuous-vs-bouts | dandiset | reproduced |
| fig1cef-representative-values | dandiset | blocked: metadata insufficient |
| fig2d, fig2g, fig2h, fig2k, fig2l, fig3e, fig3cd, fig4f, fig4hi | dandiset | blocked: data missing |

**How the analysis differs from the paper's.** The stimulus names come from the paper's code and
the stimulus software, not from the dandiset. The trial processing follows the paper's notebook
`Fig1-S1.ipynb`, except that DC files have no `vel` column, so the code's filter on it is not
applied. The dandiset has 25 ZF, not 26. Angular acceleration follows the Methods' definition, on
heading resampled to 30 Hz; the code's quantity is a variant. The test of "following the direction"
and the measure of continuous swimming are choices made here, as the paper gives no test for the
first and bases the second on bout files that are not in the dandiset.

**What was found about the dandiset.**

- It holds only the freely swimming behaviour of Figure 1 and Figure S1. The imaging and head-fixed
  tail tracking of Figures 2 to 4 are not in it, and the GitLab repository the paper names for
  "all behavioral and imaging data" holds notebooks only.
- The stimulus codes 0 to 20 have no names or description in the files or the metadata. The paper's
  code and the stimulus software disagree on which of codes 9 and 11 is lateral right.
- Subject ids 0 to 11 are each used for one ZF and one DC; the archive's summary counts 25 subjects
  and lists only *Danio rerio*.
- `f0_vtheta` is a heading change per frame, not radians/s as stated; in most DC files its sum does
  not match the change of `f0_theta`.
- 18 of 25 ZF files and 1 DC file are at 32 to 35 Hz, not above 150 Hz as the Methods say; their
  timestamp intervals and trial lengths show these are the real acquisition rates.
- No spatial scale is given; positions in pixels are consistent with the code's 10 pixels per mm,
  except one DC file whose positions reach 1490 pixels.
- ZF and DC files were exported differently: ZF files keep untracked and between-trial frames,
  DC files hold only tracked trial frames and an extra `dist` series.
- The paper's own Tukey output finds three DC pairs significant (static against backward left,
  backward right and forward left), as here, while the paper states that DC do not modulate speed.
- The stored `dist` series of DC files does not agree with the positions; the code's speed check on
  it passes trials whose speed over a short motion phase is above 100 mm/s.
- The paper's code computes "angular acceleration" as heading change per frame divided by the
  file's mean frame interval, which the gaps between trials inflate in DC files; the Figure 1H
  result depends on it. With the Methods' definition, DC values are as high at rest as during
  motion, which suggests a tracking-noise floor.